# ContractRisk — prueba visual 03 de extremo a extremo

GPU T4, sesión nueva. Diez casos con revisión humana registrada; referencias separadas de la inferencia. Ejecutar en orden. No cambia el Space ni main.

Carga VLM y generador juntos y realiza diez lecturas nuevas, una por solicitud. Una solicitud histórica de calentamiento queda excluida. Se registra tiempo de carga por separado, latencia sincronizada y memoria CUDA. No hay cuantización ni fallback automático: si no cabe, se conserva el error. No es todavía el scorecard final ni una comparación de eficiencia.

Antes de ejecutar, habilitar el acceso de este notebook al secreto `WANDB_API_KEY` en el panel de secretos. No introducir claves en código ni salidas. La prueba valida W&B antes de cargar modelos; los eventos no incluyen preguntas, OCR ni imágenes.


In [1]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_visual03_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


Commit: 0935f2037eb44ca5c135a70ce4b3923386c9950b
Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.


In [2]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-integration.txt'], 'installation.log')
import torch
run_visible([sys.executable, '-c', 'from final.run_integration import fixture_font_path; print(fixture_font_path().name)'], 'font_check.log')
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 32.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.2 MB/s

In [3]:
RESULTS = ROOT / 'visual03_results'
run_visible([sys.executable, '-u', 'final/visual_probe.py', '--output', str(RESULTS), '--require-wandb'], 'visual03.log')


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.
wandb: Currently logged in as: jcramonp (jcramonp-eanit) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: setting up run bj9xurnw
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /content/contractrisk_final_visual03_liqpfsgb/repo/wandb/run-20261117_034211-bj9xurnw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vivid-surf-4
wandb: ⭐️ View project at https://forge.coreweave.com/wandb/jcramonp-eanit/contractrisk-final
wandb: 🚀 View run at https://forge.coreweave.com/wandb/jcramonp-eanit/contractrisk-final/runs/bj9xurnw
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 

## Diagnóstico posterior, separado de la inferencia

El proceso siguiente carga las referencias después de generar respuestas. Mide coincidencia de campos, presencia literal de modalidad/valor, citas y abstención. No es juez semántico ni cumplimiento de dominio final. Todas las respuestas requieren auditoría.


In [4]:
if (RESULTS / 'predictions.jsonl').exists():
    run_visible([sys.executable, '-u', 'final/score_visual_probe.py', '--results', str(RESULTS)], 'visual03_diagnostics.log')
else:
    print('No se generaron predicciones: revisar metadata y visual03.log; conservar la evidencia del fallo.')


{
  "protocol": "visual-probe-1",
  "expected_count": 10,
  "recorded_count": 10,
  "completed_count": 10,
  "exact_extraction_count": 7,
  "literal_response_proxy_count": 9,
  "abstention_control_count": 2,
  "abstention_control_matches": 2,
  "pairs": [
    {
      "id": "vis_09_contrast",
      "base": "vis_01",
      "both_extractions_exact": false,
      "both_response_proxies_pass": true
    },
    {
      "id": "vis_10_rotation",
      "base": "vis_02",
      "both_extractions_exact": true,
      "both_response_proxies_pass": true
    }
  ],
  "observed_wall_p50_seconds": 13.21185575100003,
  "observed_wall_p95_seconds": 14.556481220900025,
  "latency_count": 10,
  "quality_approved": false,
  "warning": "Literal response proxy is not semantic correctness, independent LLM judging, or final domain compliance. Small paired sample; percentiles unstable. Blank guard not VLM success. Review all outputs and traces.",
  "predictions_sha256": "a6793c66bfe6a012951863c8f5921e4f796292c79f7

## Inspección y descarga de evidencia

Estas celdas también permiten conservar fallos. El ZIP incluye respuestas, trazas, metadata y el paquete visual revisado. Las latencias de guard y las de inferencia se identifican por caso; no atribuir una abstención determinista al VLM.


In [5]:
for name in ['metadata.json', 'diagnostics_v1.json']:
    path = RESULTS / name
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        print(name, json.dumps(data.get('summary', data), ensure_ascii=False, indent=2))


metadata.json {
  "status": "completed_pending_scoring_and_review",
  "purpose": "fresh_visual_integration_probe_not_final_scorecard",
  "protocol": "visual-probe-1",
  "integration_version": "final-integration-dev-0.2",
  "seed": 42,
  "models": {
    "embedding": [
      "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    ],
    "generator": [
      "Qwen/Qwen2.5-3B-Instruct",
      "aa8e72537993ba99e69dfaafa59ed015b17504d1"
    ],
    "reranker": [
      "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "1427fd652930e4ba29e8149678df786c240d8825"
    ],
    "vision": [
      "Qwen/Qwen2-VL-2B-Instruct",
      "895c3a49bc3fa70a340399125c650a463535e71c"
    ],
    "classifier": [
      "BSC-LT/RoBERTalex",
      "2a1c89fd468a362368463b4126751fdd51c4d847"
    ]
  },
  "started_utc": "2026-11-17T03:42:09.993727+00:00",
  "quality_approved": false,
  "references_loaded_by_inference": false,
  "wandb_status": "finished_check

In [6]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_visual_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/visual_candidates_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_visual03_resultados'), 'zip', RESULTS)
files.download(package)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>